# Analysis 3 — Step 1: 채널 피처 (OOF target encoding)

- **목적**: 누수 없는 업로드시점 채널 피처 생성. **핵심 방법론**: out-of-fold 평균으로 same-video 누수 차단.
- **피처**: `chan_mean_oof`(OOF 채널평균) · `chan_freq`(등장빈도) · `chan_mean_naive`(누수버전, Step 2 비교용)
- **fallback**: 희소/단일 채널 → 카테고리 평균 → 글로벌 평균
- **입력**: `step0_channel_merge.csv` · **산출**: `step1_channel_features.csv`

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

RANDOM_STATE, N_FOLDS = 42, 5

In [2]:
df = pd.read_csv("step0_channel_merge.csv")
vc = df["channel_title"].value_counts()
print(df.shape, "| 고유 채널:", df["channel_title"].nunique())

(6249, 27) | 고유 채널: 2100


In [3]:
# (1) chan_freq — 시점 정당한 빈도 프록시 / (2) chan_mean_naive — 전체평균(누수, 비교용)
df["chan_freq"] = df["channel_title"].map(vc).astype(int)
naive_mean = df.groupby("channel_title")["log_views"].mean()
df["chan_mean_naive"] = df["channel_title"].map(naive_mean)

In [4]:
# (3) chan_mean_oof — KFold OOF (자기 타깃 미포함) + fallback(카테고리→글로벌)
oof = np.full(len(df), np.nan)
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
fb_cat = fb_global = 0
for tr_idx, te_idx in kf.split(df):
    tr, te = df.iloc[tr_idx], df.iloc[te_idx]
    ch_mean = tr.groupby("channel_title")["log_views"].mean()      # train fold만
    cat_mean_tr = tr.groupby("category")["log_views"].mean()
    glob_tr = tr["log_views"].mean()
    for pos, (_, row) in zip(te_idx, te.iterrows()):
        ch = row["channel_title"]
        if ch in ch_mean.index and ch != "__UNKNOWN__":
            oof[pos] = ch_mean[ch]
        elif row["category"] in cat_mean_tr.index:
            oof[pos] = cat_mean_tr[row["category"]]; fb_cat += 1
        else:
            oof[pos] = glob_tr; fb_global += 1
df["chan_mean_oof"] = oof
print(f"fallback — 카테고리평균: {fb_cat}, 글로벌: {fb_global} "
      f"(총 {fb_cat + fb_global}/{len(df)} = {(fb_cat + fb_global) / len(df):.2%})")

fallback — 카테고리평균: 1434, 글로벌: 0 (총 1434/6249 = 22.95%)


In [5]:
# 점검: OOF vs naive vs 타깃 상관 (naive가 부풀려짐 = 누수 신호)
print("chan_mean_oof  vs naive 상관    :", round(df[["chan_mean_oof", "chan_mean_naive"]].corr().iloc[0, 1], 4))
print("chan_mean_oof  vs log_views 상관:", round(df[["chan_mean_oof", "log_views"]].corr().iloc[0, 1], 4))
print("chan_mean_naive vs log_views 상관(낙관편향):", round(df[["chan_mean_naive", "log_views"]].corr().iloc[0, 1], 4))
print(df[["chan_mean_oof", "chan_mean_naive", "chan_freq"]].describe().round(3).to_string())

chan_mean_oof  vs naive 상관    : 0.7176
chan_mean_oof  vs log_views 상관: 0.5513
chan_mean_naive vs log_views 상관(낙관편향): 0.8653
       chan_mean_oof  chan_mean_naive  chan_freq
count       6249.000         6249.000   6249.000
mean          13.206           13.045     14.463
std            1.224            1.571     19.130
min            7.482            6.558      1.000
25%           12.687           12.298      2.000
50%           13.172           13.158      6.000
75%           13.904           13.987     18.000
max           18.284           19.233     84.000


In [6]:
# 저장
cols = ["video_id", "channel_title", "category", "log_views",
        "chan_mean_oof", "chan_mean_naive", "chan_freq"]
df[cols].to_csv("step1_channel_features.csv", index=False)
print("[저장] step1_channel_features.csv", df[cols].shape)

[저장] step1_channel_features.csv (6249, 7)


## 점검 메모

- fallback **23%**(전부 카테고리평균; 단일 등장 채널이 자기 fold에 없을 때 발동).
- `chan_mean_oof` vs `log_views` 상관 **0.55**(강력) / naive **0.87** → OOF가 same-video 누수를 깎은 만큼 정직하게 낮음. 격차는 Step 2에서 R²로 정량화.